<a href="https://colab.research.google.com/github/Gaurav9571/week1_Gaurav_Mittal/blob/main/WEEK7_GAURAV_MITTAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip -q install -U \
langchain \
langchain-community \
langchain-google-genai \
langchain-text-splitters \
pypdf \
faiss-cpu \
sentence-transformers

In [ ]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

/tmp/ipykernel_24508/1590209351.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
os.environ["GOOGLE_API_KEY"] = input("Enter your Gemini API Key: ")

Enter your Gemini API Key: AQ.Ab8RN6IU2ieW2SzG8qO6tucfb95gLfP6B4R6gvvi8eEdbeOfUQ


In [ ]:
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded:", pdf_path)

Saving M.L. UNIT 01.pdf to M.L. UNIT 01 (1).pdf
Uploaded: M.L. UNIT 01 (1).pdf


In [ ]:
loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 22


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 73


In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_24508/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

In [ ]:
question = input("Ask a Question: ")

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

Ask a Question: give me the suummary and some questions from the pdf


In [ ]:
prompt = f"""
You are a helpful AI assistant.

Answer ONLY from the provided context.

If the answer is not available,
say:

"I couldn't find this information in the document."

Context:

{context}

Question:

{question}

Answer:
"""

response = llm.invoke(prompt)

print("\nAnswer:\n")
print(response.content)


Answer:

**Summary:**

The provided text emphasizes the critical importance of understanding the problem and its purpose in any process, stating that good results depend on this understanding. It introduces Machine Learning as a rapidly growing technology that enables computer systems to learn automatically without explicit programming. The Machine Learning life cycle is described as a cyclic process aimed at finding solutions to problems or projects, with "Test Model" mentioned as a component.

**Questions:**

1.  What is considered the most important thing in the complete process?
2.  What is the main purpose of the machine learning life cycle?
3.  How does machine learning enable computer systems to learn?
4.  What institution is mentioned in the document?
